# Distributed Representations

Study how information is encoded **across multiple dimensions**.
Is a concept linearly represented? How many dimensions does it use?
How does its encoding evolve through layers?

This notebook covers the `irtk.distributed_reps` module.

## Why This Matters

Distributed representation analysis probes how concepts are encoded across multiple dimensions rather than single neurons. Linear probes, representation rank, and cross-layer tracking reveal how information is distributed and transformed through the network.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import distributed_reps

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Linear Representation Probe

Validate whether a concept is linearly separable in activation space.

In [ ]:
# Create a simple concept: city vs non-city tokens
city_prompts = [
    "The capital of France is",
    "London is the largest city in",
    "Berlin is located in",
    "Tokyo is the capital of",
    "New York is a major city",
]
non_city_prompts = [
    "The weather today is very",
    "My favorite food is",
    "The color of the sky",
    "Mathematics is a beautiful",
    "Running is good exercise",
]

hook = "blocks.8.hook_resid_post"
all_acts = []
for prompt in city_prompts + non_city_prompts:
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(tokens)
    all_acts.append(np.array(cache[hook][-1]))  # last position

activations = np.stack(all_acts)
labels = np.array([0]*5 + [1]*5)  # 0=city, 1=non-city

result = distributed_reps.linear_representation_probe(activations, labels)
print(f"Probe accuracy: {result['accuracy']:.2f}")
print(f"Direction shape: {result['directions'].shape}")
print(f"Explained variance: {result['explained_variance']}")

## 2. Cross-Layer Concept Tracking

How does the concept's linear representation evolve through layers?

In [ ]:
token_seqs = [model.to_tokens(p) for p in city_prompts + non_city_prompts]

result = distributed_reps.cross_layer_concept_tracking(model, token_seqs, labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy by layer
axes[0].bar(range(len(result['layer_accuracies'])), result['layer_accuracies'], color='steelblue')
axes[0].set_xticks(range(len(result['labels'])))
axes[0].set_xticklabels(result['labels'], rotation=45, ha='right')
axes[0].set_ylabel('Probe accuracy')
axes[0].set_title('Concept Probe Accuracy by Layer')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)

# Direction similarity
axes[1].plot(result['direction_similarities'], 'bo-')
axes[1].set_xlabel('Layer transition')
axes[1].set_ylabel('Cosine similarity')
axes[1].set_title('Concept Direction Stability')

plt.tight_layout()
plt.show()

## 3. Writing-Reading Decomposition

How do attention heads interact with concept directions?

In [ ]:
# Use the concept direction from the probe
direction = result['layer_directions'][-1]  # last layer direction

fig, axes = plt.subplots(model.cfg.n_layers, 1, figsize=(10, 3*model.cfg.n_layers))

for layer in range(model.cfg.n_layers):
    wr = distributed_reps.writing_reading_decomposition(model, layer, direction)
    x = np.arange(model.cfg.n_heads)
    width = 0.25
    ax = axes[layer] if model.cfg.n_layers > 1 else axes
    ax.bar(x - width, wr['writing_scores'][:, 0], width, label='Writing', color='steelblue')
    ax.bar(x, wr['reading_q_scores'][:, 0], width, label='Reading (Q)', color='coral')
    ax.bar(x + width, wr['reading_k_scores'][:, 0], width, label='Reading (K)', color='green')
    ax.set_xlabel('Head')
    ax.set_ylabel('Score')
    ax.set_title(f'Layer {layer}')
    if layer == 0:
        ax.legend()

plt.suptitle('Head Interaction with Concept Direction', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Token Geometry

Visualize how specific tokens are arranged in representation space.

In [ ]:
# Compare city-related tokens
city_tokens = [tokenizer.encode(t)[0] for t in [" Paris", " London", " Berlin", " Tokyo", " Rome"]]
city_names = [" Paris", " London", " Berlin", " Tokyo", " Rome"]

result = distributed_reps.token_geometry(
    model, city_tokens, "blocks.8.hook_resid_post", token_seqs
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].imshow(result['distance_matrix'], cmap='Blues', aspect='auto')
axes[0].set_xticks(range(len(city_names)))
axes[0].set_xticklabels(city_names, rotation=45, ha='right')
axes[0].set_yticks(range(len(city_names)))
axes[0].set_yticklabels(city_names)
axes[0].set_title('Pairwise L2 Distance')
plt.colorbar(im, ax=axes[0])

im = axes[1].imshow(result['cosine_matrix'], cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
axes[1].set_xticks(range(len(city_names)))
axes[1].set_xticklabels(city_names, rotation=45, ha='right')
axes[1].set_yticks(range(len(city_names)))
axes[1].set_yticklabels(city_names)
axes[1].set_title('Cosine Similarity')
plt.colorbar(im, ax=axes[1])

plt.suptitle('City Token Geometry in Layer 8 Residual Stream', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

| Function | Purpose |
|----------|--------|
| `linear_representation_probe()` | Validate linear encoding + extract direction |
| `representation_rank()` | How many dimensions encode a concept |
| `cross_layer_concept_tracking()` | Track concept across layers |
| `writing_reading_decomposition()` | Head reading/writing to concept directions |
| `token_geometry()` | Pairwise distances and angles between tokens |